[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-fraud.ipynb)

# Full Project: Credit Card Fraud Detection

*AIBits Academy · Machine Learning End To End · Full Project*

A complete pipeline on the single most extreme class imbalance in this course — 0.167% positive class — and a hands-on demonstration of exactly why accuracy lies to you at this scale.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

def fetch(url, target, member=None):   # public source; a zip member is extracted and renamed to `target`
    if os.path.exists(target):
        return
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    blob = urllib.request.urlopen(req, timeout=120).read()
    if member:
        blob = zipfile.ZipFile(io.BytesIO(blob)).read(member)
    open(target, 'wb').write(blob)
    print('downloaded', target)

fetch('https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv', 'creditcard.csv')

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

> **Business Problem**
>
> A card issuer needs to flag fraudulent transactions in near-real-time. Missing fraud (a false negative) costs the issuer and the customer directly. But declining a legitimate transaction (a false positive) damages customer trust and can cost a customer relationship — so the model needs to be judged on the trade-off between these two error types, not on a single headline number.

> **Dataset**
>
> **284,807 European cardholder transactions, 30 features, extreme imbalance.** Features `V1`–`V28` are PCA-anonymised (the original transaction details are confidential), plus `Time` and `Amount`. Target `Class`: 0 = legitimate, 1 = fraud. After removing 1,081 duplicate rows: **283,253 legitimate vs 473 fraudulent — a 0.167% positive rate.**

## Step 1 — Load and Confirm the Imbalance

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, f1_score, accuracy_score

data = pd.read_csv("creditcard.csv")
print(data.shape)
print("Duplicate rows:", data.duplicated().sum())

data = data.drop_duplicates()
print(data['Class'].value_counts())

473 fraud cases in 283,726 transactions is a **0.167% positive rate** — by far the most extreme imbalance encountered in this course, and a genuine stress-test for everything covered on the Handling Imbalanced Data page.

## Step 2 — Scale, Split, and Stratify

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = data.drop(columns=['Class'])
y = data['Class']

X_scaled = StandardScaler().fit_transform(X)

# stratify=y is essential: without it, a random split could easily
# put almost all 473 fraud cases into the training fold and leave
# the test fold with too few to evaluate meaningfully
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y
)

## Step 3 — Why Accuracy Lies: Logistic Regression Baseline

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("F1 (fraud class):", f1_score(y_test, y_pred))

> **99.91% Accuracy, 58% Fraud Recall**
>
> The headline number screams "excellent model." The confusion matrix tells the real story: of the 95 fraud cases in the test set, only 58% (55 of them) were caught — 40 fraudulent transactions were waved through. Accuracy is dominated by the 56,651 easy, correctly-classified legitimate transactions and barely moves regardless of how well the model handles the 95 that actually matter to the business. This is *the* canonical illustration of why accuracy is the wrong headline metric under severe imbalance — covered in principle on the Model Evaluation page, and here with real numbers attached.

## Step 4 — Random Forest and XGBoost: Better Fraud Recall at the Same Accuracy

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

rf = RandomForestClassifier()
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("Random Forest F1 (fraud):", f1_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

xg = XGBClassifier(n_jobs=-1)
xg.fit(X_train, y_train)
y_pred_xg = xg.predict(X_test)
print("XGBoost F1 (fraud):", f1_score(y_test, y_pred_xg))
print(classification_report(y_test, y_pred_xg))

Both ensembles improve fraud-class F1 substantially (0.69 → 0.84 → 0.85) at essentially the same 99.9%+ accuracy — accuracy was never the metric distinguishing these three models; F1 (or precision/recall directly) was, the entire time.

## Step 5 — Does SMOTE Help? A Genuine Trade-Off, Not a Free Win

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE()
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(np.bincount(y_train_sm))   # now perfectly balanced

xg_smote = XGBClassifier(n_jobs=-1)
xg_smote.fit(X_train_sm, y_train_sm)
y_pred_sm = xg_smote.predict(X_test)
print("SMOTE + XGBoost F1 (fraud):", f1_score(y_test, y_pred_sm))
print(classification_report(y_test, y_pred_sm))

| Model | Fraud Precision | Fraud Recall | Fraud F1 |
|---|---|---|---|
| Logistic Regression | 0.85 | 0.58 | 0.69 |
| Random Forest | 0.97 | 0.74 | 0.84 |
| **XGBoost (no SMOTE)** | 0.97 | 0.75 | **0.85 (best)** |
| XGBoost + SMOTE | 0.76 | 0.79 | 0.77 |

> **The SMOTE Trade-Off, Concretely**
>
> SMOTE *did* do what it's supposed to: recall rose (0.75 → 0.79), catching 4 more fraud cases out of 95. But precision fell sharply (0.97 → 0.76), meaning a much larger share of transactions the model flags as fraud are actually legitimate — more false declines, more customer friction, more manual review workload for the fraud team. The net F1 is *lower* with SMOTE (0.77 vs 0.85). Neither number is "correct" in isolation: which model is actually better depends on the issuer's real cost ratio between a missed fraud and a wrongly-declined legitimate purchase — a business decision, not a modelling one. This is the exact nuance the Handling Imbalanced Data page's oversampling section warns about in the abstract; here it plays out in real numbers.

## Visualizing the Precision/Recall Trade-Off

Fraud-class precision, recall, and F1 across all four approaches — watch SMOTE trade precision for recall, netting a *lower* F1 despite catching more fraud.

## Key Business Takeaways

- Under 0.167% positive-class imbalance, accuracy is nearly useless as a headline metric — a model that predicts "never fraud" scores 99.83% accuracy while catching zero fraud.
- Stratified train/test splitting is not optional at this level of imbalance — an unstratified split can leave a test set with too few positive examples to evaluate reliably.
- Tree ensembles (Random Forest, XGBoost) substantially outperformed Logistic Regression on fraud-class F1 here, without any resampling.
- SMOTE oversampling traded precision for recall and produced a *lower* F1 on this dataset — oversampling should be tested, not assumed to help, and the right choice ultimately depends on the business's relative cost of false positives vs false negatives.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · How rare is fraud?

Store in `fraud_rate` the fraction of transactions with `Class == 1` in the de-duplicated `data`, and in `never_fraud_acc` the accuracy of a model that always predicts "legitimate".

In [ ]:
fraud_rate = never_fraud_acc = None   # TODO


In [ ]:
try:
    check("about 0.167%", abs(fraud_rate - 473 / 283726) < 1e-9)
    check("accuracy floor", abs(never_fraud_acc - (1 - fraud_rate)) < 1e-12)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
fraud_rate = float(data["Class"].mean())
never_fraud_acc = 1 - fraud_rate

```

</details>

### Exercise 2 · Medium · Precision and recall from the confusion matrix

For the logistic-regression predictions (`y_pred` from Step 3 - recompute with `lr.predict(X_test)` if you have run later cells), compute `tp`, `fp`, `fn` with `confusion_matrix`, then `prec_lr` and `rec_lr`.

In [ ]:
tp = fp = fn = prec_lr = rec_lr = None   # TODO


In [ ]:
try:
    from sklearn.metrics import precision_score, recall_score
    yp = lr.predict(X_test)
    check("precision", abs(prec_lr - precision_score(y_test, yp)) < 1e-9)
    check("recall", abs(rec_lr - recall_score(y_test, yp)) < 1e-9)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
yp = lr.predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, yp).ravel()
prec_lr = tp / (tp + fp)
rec_lr = tp / (tp + fn)

```

</details>

### Exercise 3 · Stretch · Lower the threshold, keep precision above 90%

With `rf.predict_proba(X_test)[:, 1]`, find the **lowest** threshold (searching 0.01, 0.02, ..., 0.99) whose fraud precision is still at least 0.90. Store it in `thr` and the recall at that threshold in `rec_at_thr`.

In [ ]:
thr = rec_at_thr = None   # TODO


In [ ]:
try:
    from sklearn.metrics import recall_score, precision_score
    p = rf.predict_proba(X_test)[:, 1]
    check("precision target met", precision_score(y_test, p >= thr) >= 0.90)
    check("recall reported", abs(rec_at_thr - recall_score(y_test, p >= thr)) < 1e-9)
    check("recall is no worse than at 0.5", rec_at_thr >= recall_score(y_test, p >= 0.5))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
p = rf.predict_proba(X_test)[:, 1]
ok = [t for t in np.arange(0.01, 1.0, 0.01) if (p >= t).sum() > 0 and precision_score(y_test, p >= t) >= 0.90]
thr = float(min(ok))
rec_at_thr = recall_score(y_test, p >= thr)

```

Lowering the threshold trades precision for recall - the right point depends on what a missed fraud costs versus a false alarm.

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: Credit Card Fraud Detection**.*